# verify_csvs - integrity check for per-epoch NC CSVs

Recompute `T_NC`, `fn*`, CV, 95% Student-t CI, and 95% bootstrap CI directly
from the per-epoch CSVs produced by the training notebooks (E1 / E2 / E4) or
the original published runs.


**Definition (must match the paper).** `T_NC` is the *first epoch in
Phase 2* at which `NC1 < threshold`. Phase-1 NC1 dips (visible e.g. with
Tanh during CE training) are transient and are not equilibrium collapses.
The script applies this Phase-2-only filter whenever a `phase` column is
present in the CSV.

**Inputs.** A directory containing per-epoch CSVs named like
`fmnist_s0.csv`, `cifar100_s1.csv`, `actLeakyReLU_s2.csv`, etc.
(general pattern: `<condition>_s<seed>.csv`).

**Dependencies.** `pandas`, `numpy` only - no `torch`.

Works on Colab, Kaggle, or a local machine.


In [1]:
# Pick the directory to verify. Set this before running the cells below.
# Examples:
#   Colab        : '/content/'
#   Kaggle       : '/kaggle/working/'
#   Local Linux  : '/home/me/Downloads/Results/'
#   Local macOS  : '/Users/me/Downloads/NC-paper-IEEE-Access/working_files/source-and-notebooks/results/'
DIRECTORY = '../results'        # <- edit me

# Which NC1 thresholds to report.
THRESHOLDS = (0.01, 0.05)


In [2]:
import os, re, sys
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

def _bootstrap_ci(values, n_resamples=10_000, ci=0.95, rng_seed=0):
    if len(values) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(rng_seed)
    idx = rng.integers(0, len(values), size=(n_resamples, len(values)))
    means = values[idx].mean(axis=1)
    alpha = (1.0 - ci) / 2.0
    return (float(np.quantile(means, alpha)),
            float(np.quantile(means, 1.0 - alpha)))

def _t_interval(values, ci=0.95):
    n = len(values)
    if n < 2:
        return (np.nan, np.nan)
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(n)
    # 95% two-sided t-multipliers for small N (avoids scipy dependency).
    t_table = {2: 12.706, 3: 4.303, 4: 3.182, 5: 2.776,
               6: 2.571, 7: 2.447, 8: 2.365, 9: 2.306, 10: 2.262}
    t = t_table.get(n, 1.96)
    return (float(mean - t * se), float(mean + t * se))

def _find_tnc(df, nc1_threshold, phase2_only=True):
    if 'nc1' not in df.columns or 'feat_norm' not in df.columns:
        return (None, None)
    sub = df
    if phase2_only and 'phase' in df.columns:
        sub = df[df['phase'] == 2]
    mask = sub['nc1'].notna() & (sub['nc1'] < nc1_threshold)
    if not mask.any():
        return (None, None)
    first = sub[mask].iloc[0]
    epoch = int(first['epoch']) if pd.notna(first['epoch']) else None
    fn = float(first['feat_norm']) if pd.notna(first['feat_norm']) else None
    return (epoch, fn)

def _classify_files(csv_files):
    groups = defaultdict(list)
    summaries, other = [], []
    pattern = re.compile(r'^(.*)_s(\d+)\.csv$')
    for p in csv_files:
        m = pattern.match(p.name)
        if m:
            groups[m.group(1)].append((int(m.group(2)), p))
        elif p.name.endswith('_summary.csv') or 'summary' in p.name.lower():
            summaries.append(p)
        else:
            other.append(p)
    for k in groups:
        groups[k].sort()
    return groups, summaries, other

def summarise_group(prefix, members, thresholds):
    print(f'\n=== {prefix} (N seeds found = {len(members)}) ===')
    per_seed = []
    for seed, path in members:
        try:
            df = pd.read_csv(path)
        except Exception as exc:
            print(f'  seed {seed}: cannot read {path.name} ({exc})')
            continue
        n_rows = len(df)
        n_term = int(df['nc1'].notna().sum()) if 'nc1' in df.columns else 0
        row = {'seed': seed, 'file': path.name,
               'n_epochs_logged': n_rows, 'n_terminal_rows': n_term}
        for thr in thresholds:
            tnc, fn = _find_tnc(df, thr)
            row[f'T_NC@{thr}'] = tnc
            row[f'fn@{thr}']   = fn
        per_seed.append(row)

    if not per_seed:
        print('  no readable CSVs')
        return None

    pdf = pd.DataFrame(per_seed)
    print(pdf.to_string(index=False))

    for thr in thresholds:
        col = f'fn@{thr}'
        ok = pdf[col].dropna().values.astype(float)
        if len(ok) >= 2:
            mean = ok.mean(); std = ok.std(ddof=1)
            cv = 100.0 * std / mean if mean != 0 else float('nan')
            t_lo, t_hi = _t_interval(ok)
            b_lo, b_hi = _bootstrap_ci(ok)
            print(f'  fn at NC1<{thr}: N={len(ok)}  '
                  f'mean={mean:.4f}  std={std:.4f}  CV={cv:.2f}%')
            print(f'    95% t-CI       [{t_lo:.4f}, {t_hi:.4f}]')
            print(f'    95% bootstrap  [{b_lo:.4f}, {b_hi:.4f}]')
        elif len(ok) == 1:
            print(f'  fn at NC1<{thr}: only N=1 seed (fn={ok[0]:.4f}); '
                  f'cannot compute CV / CI')
        else:
            print(f'  fn at NC1<{thr}: 0 seeds reached this threshold')
    return pdf

print('helpers defined.')


helpers defined.


In [3]:
root = Path(DIRECTORY)
assert root.is_dir(), f'not a directory: {root}'

csv_files = sorted(root.glob('*.csv'))
if not csv_files:
    print(f'no CSVs found in {root}')
else:
    groups, summaries, other = _classify_files(csv_files)
    print(f'Found {len(csv_files)} CSVs: '
          f'{len(groups)} per-seed groups, '
          f'{len(summaries)} summary files, {len(other)} other')
    group_pdfs = {}
    for prefix in sorted(groups):
        group_pdfs[prefix] = summarise_group(prefix, groups[prefix], THRESHOLDS)
    if summaries:
        print('\n=== Summary file contents (for visual diff) ===')
        for s_path in summaries:
            try:
                s = pd.read_csv(s_path)
                print(f'\n  {s_path.name}:')
                print(s.to_string(index=False))
            except Exception as exc:
                print(f'  {s_path.name}: read error ({exc})')
    print('\nDone.')


Found 100 CSVs: 25 per-seed groups, 11 summary files, 19 other

=== actGELU (N seeds found = 3) ===
 seed           file  n_epochs_logged  n_terminal_rows T_NC@0.01 fn@0.01  T_NC@0.05  fn@0.05
    0 actGELU_s0.csv               60               60      None    None        240 2.078021
    1 actGELU_s1.csv               60               60      None    None        250 1.897165
    2 actGELU_s2.csv               60               60      None    None        250 1.589721
  fn at NC1<0.01: 0 seeds reached this threshold
  fn at NC1<0.05: N=3  mean=1.8550  std=0.2469  CV=13.31%
    95% t-CI       [1.2417, 2.4683]
    95% bootstrap  [1.5897, 2.0780]

=== actLeakyReLU (N seeds found = 3) ===
 seed                file  n_epochs_logged  n_terminal_rows  T_NC@0.01  fn@0.01  T_NC@0.05  fn@0.05
    0 actLeakyReLU_s0.csv               70               70        250 0.995722        220 1.411079
    1 actLeakyReLU_s1.csv               70               70        300 1.026419        210 1.692758
    2 a